# Hey Igor microWakeWord Training

Clean Colab notebook for training and exporting an ESPHome-compatible `micro_wake_word` model.

Notes:
- Background audio prep avoids `datasets.Audio()` decoding to stay compatible with current Colab runtimes.
- Positive examples are trained through `type: "clips"`.
- Built-in validation/export in this `micro-wake-word` version is not reliable with `clips`-based datasets, so training and export are split.
- For a real training run, use a Colab GPU runtime.


In [ ]:
# 1. Test one generated pronunciation

import os
import sys
import importlib.util
from functools import wraps
from IPython.display import Audio

if not os.path.exists("./piper-sample-generator"):
    !git clone https://github.com/rhasspy/piper-sample-generator
    !cd piper-sample-generator && git checkout 213d4d5

model_path = "piper-sample-generator/models/en_US-libritts_r-medium.pt"
if not os.path.exists(model_path):
    !wget -O {model_path} "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"

!pip install -q piper-tts piper-phonemize-cross webrtcvad

if importlib.util.find_spec("torch") is None:
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu

import torch

_original_torch_load = torch.load

@wraps(_original_torch_load)
def _torch_load_compat(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_torch_load(*args, **kwargs)

torch.load = _torch_load_compat

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")

from generate_samples import generate_samples

target_word = "hey___ee__gore"  # @param {type:"string"}

def text_to_speech(text):
    generate_samples(
        text=text,
        max_samples=1,
        length_scales=[1.1],
        noise_scales=[0.7],
        noise_scale_ws=[0.7],
        output_dir="./",
        batch_size=1,
        auto_reduce_batch_size=True,
        file_names=["test_generation.wav"],
    )

text_to_speech(target_word)
Audio("test_generation.wav", autoplay=True)


In [ ]:
# 2. Install microWakeWord and dependencies

import locale
import os
import site

def getpreferredencoding(do_setlocale=True):
    return "UTF-8"

locale.getpreferredencoding = getpreferredencoding

!apt-get -qq update
!apt-get -qq install -y ffmpeg git-lfs
!pip install -q "git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f"
!pip install -q datasets scipy tqdm pyyaml

if not os.path.exists("./micro-wake-word"):
    !git clone https://github.com/OHF-Voice/micro-wake-word

!pip install -q -e ./micro-wake-word
site.main()

import microwakeword
print("microwakeword import OK:", microwakeword.__file__)


In [ ]:
# 3. Imports

import json
import os
import shutil
import subprocess
from pathlib import Path
from types import SimpleNamespace

import datasets
import numpy as np
import scipy
import yaml
from tqdm import tqdm

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")

from generate_samples import generate_samples
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
import microwakeword.data as input_data
import microwakeword.mixednet as mixednet
import microwakeword.utils as utils
from microwakeword.layers import modes
from microwakeword.model_train_eval import load_config


In [ ]:
# 4. Download RIR dataset

output_dir = Path("./mit_rirs")
source_dir = Path("./MIT_environmental_impulse_responses/16khz")

output_dir.mkdir(exist_ok=True)

if not source_dir.exists():
    !git lfs install
    !git clone https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses

rir_files = list(source_dir.glob("*.wav"))
if not rir_files:
    raise RuntimeError(f"No RIR wav files found in {source_dir}")

existing_rirs = len(list(output_dir.glob("*.wav")))
if existing_rirs != len(rir_files):
    for src in tqdm(rir_files):
        shutil.copy2(src, output_dir / src.name)

print({"mit_rirs": len(list(output_dir.glob('*.wav')))})


In [ ]:
# 5. Audio conversion helper

def convert_audio_tree(src_root, pattern, out_dir):
    src_root = Path(src_root)
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True)

    files = sorted(src_root.glob(pattern))
    if not files:
        raise RuntimeError(f"No source files found in {src_root} matching {pattern}")

    for src in tqdm(files):
        rel_stem = src.relative_to(src_root).with_suffix("")
        dst_name = "__".join(rel_stem.parts) + ".wav"
        dst = out_dir / dst_name
        if dst.exists():
            continue

        subprocess.run(
            ["ffmpeg", "-y", "-i", str(src), "-ac", "1", "-ar", "16000", str(dst)],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )

    print({out_dir.name: len(list(out_dir.glob('*.wav')))})


In [ ]:
# 6. Download AudioSet sample from parquet and convert to 16 kHz wav

output_dir = Path("./audioset_16k")
output_dir.mkdir(exist_ok=True)
tmp_dir = Path("./audioset_tmp")
tmp_dir.mkdir(exist_ok=True)

max_examples = 2000  # @param {type:"slider", min:500, max:10000, step:500}

audioset = datasets.load_dataset(
    "agkphysics/AudioSet",
    "balanced",
    split=f"train[:{max_examples}]",
)
audioset = audioset.cast_column("audio", datasets.Audio(decode=False))

for row in tqdm(audioset):
    audio = row["audio"]
    dst = output_dir / f"{row['video_id']}.wav"
    if dst.exists():
        continue

    src_path = None
    if audio.get("path") and os.path.exists(audio["path"]):
        src_path = audio["path"]
    elif audio.get("bytes") is not None:
        tmp_src = tmp_dir / f"{row['video_id']}.flac"
        tmp_src.write_bytes(audio["bytes"])
        src_path = str(tmp_src)

    if src_path is None:
        continue

    subprocess.run(
        ["ffmpeg", "-y", "-i", src_path, "-ac", "1", "-ar", "16000", str(dst)],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

print({"audioset_16k": len(list(output_dir.glob('*.wav')))})


In [ ]:
# 7. Download FMA sample and convert to 16 kHz wav

download_dir = Path("./fma")
download_dir.mkdir(exist_ok=True)
fname = "fma_xs.zip"
link = "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/" + fname
out_path = download_dir / fname

if not (download_dir / "fma_small").exists():
    if not out_path.exists():
        !wget -O {out_path} {link}
    !cd {download_dir} && unzip -q {fname}

convert_audio_tree("fma/fma_small", "**/*.mp3", "fma_16k")


In [ ]:
# 8. Generate positive training samples with Piper

os.makedirs("generated_samples", exist_ok=True)

number_of_examples = 3000  # @param {type:"slider", min:500, max:20000, step:500}

generate_samples(
    text=target_word,
    max_samples=number_of_examples,
    output_dir="generated_samples",
    batch_size=32,
    auto_reduce_batch_size=True,
)

print({"generated_samples": len(list(Path('generated_samples').glob('*.wav')))})


In [ ]:
# 9. Build reusable dataset settings and smoke-test the current API

clips_settings = {
    "input_directory": "generated_samples",
    "file_pattern": "*.wav",
    "max_clip_duration_s": None,
    "remove_silence": False,
    "random_split_seed": 10,
    "split_count": 0.1,
}

background_counts = {
    "fma_16k": len(list(Path("fma_16k").rglob("*.wav"))),
    "audioset_16k": len(list(Path("audioset_16k").rglob("*.wav"))),
}
if not all(background_counts.values()):
    raise RuntimeError(f"Background audio is missing. Counts: {background_counts}")

augmentation_settings = {
    "augmentation_duration_s": 3.2,
    "augmentation_probabilities": {
        "SevenBandParametricEQ": 0.1,
        "TanhDistortion": 0.1,
        "PitchShift": 0.1,
        "BandStopFilter": 0.1,
        "AddColorNoise": 0.1,
        "AddBackgroundNoise": 0.75,
        "Gain": 1.0,
        "RIR": 0.5,
    },
    "impulse_paths": ["mit_rirs"],
    "background_paths": ["fma_16k", "audioset_16k"],
    "background_min_snr_db": -5,
    "background_max_snr_db": 10,
    "min_jitter_s": 0.195,
    "max_jitter_s": 0.205,
}

positive_spectrogram_generation_settings = {
    "step_ms": 10,
}

negative_spectrogram_generation_settings = {
    "step_ms": 10,
}

negative_augmentation_settings = {
    "augmentation_duration_s": 3.2,
    "augmentation_probabilities": {
        "SevenBandParametricEQ": 0.0,
        "TanhDistortion": 0.0,
        "PitchShift": 0.0,
        "BandStopFilter": 0.0,
        "AddColorNoise": 0.0,
        "AddBackgroundNoise": 0.0,
        "Gain": 0.0,
        "GainTransition": 0.0,
        "RIR": 0.0,
    },
    "impulse_paths": [],
    "background_paths": [],
    "min_jitter_s": 0.0,
    "max_jitter_s": 0.0,
    "truncate_randomly": True,
}

fma_clips_settings = {
    "input_directory": "fma_16k",
    "file_pattern": "*.wav",
    "max_clip_duration_s": None,
    "remove_silence": False,
    "random_split_seed": 10,
    "split_count": 0.1,
}

audioset_clips_settings = {
    "input_directory": "audioset_16k",
    "file_pattern": "*.wav",
    "max_clip_duration_s": None,
    "remove_silence": False,
    "random_split_seed": 10,
    "split_count": 0.1,
}

clips = Clips(**clips_settings)
augmenter = Augmentation(**augmentation_settings)
spectrograms = SpectrogramGeneration(
    clips=clips,
    augmenter=augmenter,
    **positive_spectrogram_generation_settings,
)

sample_spectrogram = next(spectrograms.spectrogram_generator(random=True, max_clips=1))
print({
    "background_counts": background_counts,
    "sample_spectrogram_shape": sample_spectrogram.shape,
})


In [ ]:
# 10. Write full training config

config = {}
config["window_step_ms"] = 10
config["train_dir"] = "trained_models/hey_igor_full"
config["features"] = [
    {
        "type": "clips",
        "truth": True,
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truncation_strategy": "truncate_start",
        "clips_settings": clips_settings,
        "augmentation_settings": augmentation_settings,
        "spectrogram_generation_settings": positive_spectrogram_generation_settings,
    },
    {
        "type": "clips",
        "truth": False,
        "sampling_weight": 8.0,
        "penalty_weight": 1.0,
        "truncation_strategy": "random",
        "clips_settings": fma_clips_settings,
        "augmentation_settings": negative_augmentation_settings,
        "spectrogram_generation_settings": negative_spectrogram_generation_settings,
    },
    {
        "type": "clips",
        "truth": False,
        "sampling_weight": 8.0,
        "penalty_weight": 1.0,
        "truncation_strategy": "random",
        "clips_settings": audioset_clips_settings,
        "augmentation_settings": negative_augmentation_settings,
        "spectrogram_generation_settings": negative_spectrogram_generation_settings,
    },
]
config["training_steps"] = [2000]
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]
config["learning_rates"] = [0.001]
config["batch_size"] = 64
config["time_mask_max_size"] = [0]
config["time_mask_count"] = [0]
config["freq_mask_max_size"] = [0]
config["freq_mask_count"] = [0]
config["eval_step_interval"] = 999999
config["clip_duration_ms"] = 1500
config["target_minimization"] = 0.9
config["minimization_metric"] = None
config["maximization_metric"] = "average_viable_recall"

with open("training_parameters.yaml", "w") as f:
    yaml.dump(config, f)

print(yaml.safe_load(open("training_parameters.yaml"))["train_dir"])


In [ ]:
# 11. Train full model

import shlex

config = yaml.safe_load(open("training_parameters.yaml"))
shutil.rmtree(config["train_dir"], ignore_errors=True)

train_cmd = "python -m microwakeword.model_train_eval --training_config=training_parameters.yaml --train 1 --restore_checkpoint 0 --test_tf_nonstreaming 0 --test_tflite_nonstreaming 0 --test_tflite_nonstreaming_quantized 0 --test_tflite_streaming 0 --test_tflite_streaming_quantized 0 --use_weights last_weights mixednet --pointwise_filters 64,64,64,64 --repeat_in_block 1,1,1,1 --mixconv_kernel_sizes [5],[7,11],[9,15],[23] --residual_connection 0,0,0,0 --first_conv_filters 32 --first_conv_kernel_size 5 --stride 3"

train_result = subprocess.run(
    shlex.split(train_cmd),
    text=True,
    capture_output=True,
)

print("TRAIN RETURNCODE:", train_result.returncode)
print(train_result.stdout)
print(train_result.stderr)

weights_path = Path(config["train_dir"]) / "last_weights.weights.h5"
if not weights_path.exists():
    raise RuntimeError(f"Training did not produce weights at {weights_path}")

print("Training weights:", weights_path)


In [ ]:
# 12. Export quantized streaming TFLite

flags = SimpleNamespace(
    training_config="training_parameters.yaml",
    stride=3,
    pointwise_filters="64,64,64,64",
    repeat_in_block="1,1,1,1",
    mixconv_kernel_sizes="[5],[7,11],[9,15],[23]",
    residual_connection="0,0,0,0",
    first_conv_filters=32,
    first_conv_kernel_size=5,
    spatial_attention=0,
    max_pool=0,
    pooled=0,
)

config = load_config(flags, mixednet)
audio_processor = input_data.FeatureHandler(config)

model = mixednet.model(flags, shape=config["training_input_shape"], batch_size=1)
model.load_weights(os.path.join(config["train_dir"], "last_weights.weights.h5"))

utils.convert_model_saved(
    model,
    config,
    folder="stream_state_internal",
    mode=modes.Modes.STREAM_INTERNAL_STATE_INFERENCE,
)

utils.convert_saved_model_to_tflite(
    config,
    audio_processor=audio_processor,
    path_to_model=os.path.join(config["train_dir"], "stream_state_internal"),
    folder=os.path.join(config["train_dir"], "tflite_stream_state_internal_quant"),
    fname="stream_state_internal_quant.tflite",
    quantize=True,
)

print("TFLite artifacts:")
for p in sorted(Path(config["train_dir"]).rglob("*.tflite")):
    print("-", p)


In [ ]:
# 13. Package ESPHome model files

from google.colab import files

src_model = Path("trained_models/hey_igor_full/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite")
if not src_model.exists():
    raise RuntimeError(f"Missing model: {src_model}")

shutil.copy(src_model, "hey_igor.tflite")

manifest = {
    "type": "micro",
    "wake_word": "Hey Igor",
    "author": "Alex",
    "website": "https://github.com/cajo-dk/microwakeword",
    "model": "hey_igor.tflite",
    "trained_languages": ["en"],
    "version": 2,
    "micro": {
        "probability_cutoff": 0.97,
        "sliding_window_size": 5,
        "feature_step_size": 10,
        "tensor_arena_size": 262144,
        "minimum_esphome_version": "2024.7",
    },
}

with open("hey_igor.json", "w") as f:
    json.dump(manifest, f, indent=2)

files.download("hey_igor.tflite")
files.download("hey_igor.json")
